# Part 1 : Tools in AI Agents

## TASK 1: Understanding

1. What is a tool in an AI Agent?
   - A tool is an external function or capability that an AI agent can use to perform a specific task that the language model cannot reliably do by itself.

2. Why do agents need tools?
    - Agents need tools because an LLM alone has limitations. 
    - It mainly generates responses based on its learned knowledge and the information provided in the current context.

3. Difference between a chatbot and an agent
   - Chatbots mainly responds to user messages, usually generate text responses, follows a converstational flow, limited ability
   - AI Agents: can decide how to complete a task, can use tools and external systems, can perform multi step reasoning

# Part 2

## TASK 2: Built-in Tools in LangChain

In [73]:
from langchain_community.tools import (
WikipediaQueryRun,
    TavilySearchResults
)

In [74]:
from langchain_community.utilities import (
    WikipediaAPIWrapper
)

In [75]:
from langchain_core.tools import tool

In [76]:
from dotenv import load_dotenv

In [77]:
load_dotenv()

True

In [78]:
@tool
def calculator(expression: str) -> str:
    """Perform a basic mathematical calculation."""

    try:
        result = eval(expression)
        return str(result)

    except Exception as e:
        return f"Error: {e}"

In [79]:
wikipedia = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(
        top_k_results=2,
        doc_content_chars_max=1000
    )
)



In [80]:
web_search = TavilySearchResults(
    max_results=3
)


In [81]:
calculator.invoke("25 * 4 + 10")

'110'

In [85]:
wikipedia.invoke("Artificial Intelligence")

'Page: Artificial intelligence\nSummary: Artificial intelligence (AI) is the capability of computational systems to perform tasks typically associated with human intelligence, such as learning, reasoning, problem-solving, perception, and decision-making. It is a field of research in engineering, mathematics, and computer science that develops and studies methods and software that enable machines to perceive their environment and use learning and intelligence to take actions that maximise their chances of achieving defined goals.\nHigh-profile applications of AI include advanced web search engines, chatbots, virtual assistants, autonomous vehicles, play and analysis in strategy games (e.g., chess and Go), and content generation (e.g. images, audio, and videos).\nThe traditional goals of AI research include learning, reasoning, knowledge representation, planning, natural language processing, and perception, as well as support for robotics. To reach these goals, AI researchers use techniq

In [86]:
web_search.invoke("latest developments in Generative AI")

[{'title': 'Recent Developments in Generative AI',
  'url': 'https://www.youtube.com/watch?v=uF2a5kThM3g',
  'content': "# Recent Developments in Generative AI\n## Accenture\n93500 subscribers\n63 likes\n\n### Description\n3818 views\nPosted: 28 Mar 2025\nIt seems like every day, there is a new advance in generative AI. Large multimodal  models (LMMs) and Small Language Models (SMLs) are becoming popular, while Retrieval-Augmented Generation (RAG) is boosting the accuracy of Gen AI models. Then, there are open-source models democratizing access to Gen AI, though not all of them are available or customizable for free.\nWith all these and more, learn about the exciting evolution of Gen AI and why the future is looking very interesting indeed! [...] [Music] the era of generative AI started on November 30th 2022 with the launch of chat GPT and ever since the generative AI landscape has been expanding and evolving at a record Pace much faster than any other technology we've seen it seems li

# Part 2: Creating Custom Tools and Toolkits

## TASK 3: Create A custom tool

In [87]:
@tool
def company_policy_lookup(query: str) -> str:
    """
    Look up company policy information.
    """

    policies = {
        "leave": "Employees receive 20 paid leaves per year.",
        "work from home": "Employees can work from home 2 days per week.",
        "working hours": "Standard working hours are 9:00 AM to 6:00 PM.",
        "notice period": "The standard notice period is 30 days.",
        "remote": "Remote work is allowed with manager approval."
    }

    query = query.lower()
    for key, value in policies.items():
        if key in query:
            return value

    return "No company policy found for this query."

In [18]:
company_policy_lookup.invoke("What is the leave policy?")

'Employees receive 20 paid leaves per year.'

In [19]:
company_policy_lookup.invoke("Can employees work from home?")

'Employees can work from home 2 days per week.'

In [20]:
company_policy_lookup.invoke("What is the notice period?")

'The standard notice period is 30 days.'

## TASK 4: Create a Custom Toolkit

In [ ]:
from langchain_core.tools import tool

In [49]:
@tool
def policy_lookup(query: str) -> str:
    """Look up company policies."""

    policies = {
        "leave": "Employees receive 20 paid leaves per year.",
        "work from home": "Employees can work from home 2 days per week.",
        "working hours": "Standard working hours are 9:00 AM to 6:00 PM.",
        "notice period": "The standard notice period is 30 days."
    }

    query = query.lower()
    for key, value in policies.items():
        if key in query:
            return value

    return "Policy not found."

In [50]:
@tool
def database_query(query: str) -> str:
    """Query simple employee database."""

    employees = {
        "101": {
            "name": "Rahul",
            "department": "Engineering"
        },

        "102": {
            "name": "Priya",
            "department": "HR"
        },

        "103": {
            "name": "Amit",
            "department": "Finance"
        }
    }

    employee_id = query.strip()
    if employee_id in employees:
        employee = employees[employee_id]
        return (
            f"Employee: {employee['name']}, "
            f"Department: {employee['department']}"
        )
    return "Employee not found."

In [56]:
from datetime import datetime

@tool
def get_current_datetime() -> str:
    """Return the current date and time."""

    return datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

In [52]:
company_toolkit = [
    policy_lookup,
    database_query,
    get_current_datetime
]

In [53]:
policy_lookup.invoke("What is the work from home policy?")

'Employees can work from home 2 days per week.'

In [54]:
database_query.invoke("101")

'Employee: Rahul, Department: Engineering'

In [72]:
get_current_datetime.invoke({})

'2026-09-14 18:27:29'

# Part 3: Tool Binding and Tool Calling

## TASK 5: Tool Binding to LLM

In [31]:
import os
from datetime import datetime

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

In [33]:
@tool
def calculator(expression: str) -> str:
    """Calculate a mathematical expression."""

    try:
        result = eval(expression)
        return str(result)

    except Exception as e:
        return f"Calculation error: {e}"


In [34]:
@tool
def company_policy_lookup(query: str) -> str:
    """Look up company policies."""

    policies = {
        "leave": "Employees receive 20 paid leaves per year.",
        "work from home": "Employees can work from home 2 days per week.",
        "working hours": "Working hours are 9 AM to 6 PM.",
        "notice period": "The standard notice period is 30 days."
    }

    query = query.lower()

    for key, value in policies.items():

        if key in query:
            return value

    return "No matching company policy was found."

In [35]:
@tool
def get_current_time() -> str:
    """Return the current date and time."""

    return datetime.now().strftime(
        "%Y-%m-%d %H:%M:%S"
    )

In [36]:
tools = [
    calculator,
    company_policy_lookup,
    get_current_time
]


In [37]:
tool_map = {
    tool.name: tool
    for tool in tools
}


In [38]:
llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

In [39]:
llm_with_tools = llm.bind_tools(tools)


In [40]:
def run_agent(user_query):
    print(user_query)
    response = llm_with_tools.invoke(user_query)

    if not response.tool_calls:
        print(response.content)
        return

    for tool_call in response.tool_calls:

        print(f"Tool: {tool_call['name']}")
        print(f"Arguments: {tool_call['args']}")

    tool_results = []

    for tool_call in response.tool_calls:
        tool_name = tool_call["name"]
        tool_args = tool_call["args"]
        selected_tool = tool_map[tool_name]
        print(f"Executing: {tool_name}")
        result = selected_tool.invoke(tool_args)
        print(f"Result: {result}")
        tool_results.append(
            {
                "tool_call": tool_call,
                "result": result
            }
        )
    messages = [
        {
            "role": "user",
            "content": user_query
        },
        response
    ]


    for item in tool_results:

        messages.append(
            {
                "role": "tool",
                "tool_call_id": item["tool_call"]["id"],
                "content": item["result"]
            }
        )

    final_response = llm_with_tools.invoke(messages)

    print(final_response.content)

In [ ]:
run_agent("What is 25 multiplied by 40?")

What is 25 multiplied by 40?
Tool: calculator
Arguments: {'expression': '25 * 40'}
Executing: calculator
Result: 1000
25 multiplied by 40 is 1000.
What is the company work from home policy?
Tool: company_policy_lookup
Arguments: {'query': 'work from home policy'}
Executing: company_policy_lookup
Result: Employees can work from home 2 days per week.
The company work from home policy allows employees to work from home 2 days per week.
Calculate 5000 * 0.15 and tell me the current date and time.
Tool: calculator
Arguments: {'expression': '5000 * 0.15'}
Tool: get_current_time
Arguments: {}
Executing: calculator
Result: 750.0
Executing: get_current_time
Result: 2026-09-14 18:10:54
The result of \( 5000 \times 0.15 \) is \( 750.0 \).

The current date and time is September 14, 2026, at 18:10:54.


In [ ]:
run_agent("What is the company work from home policy?")

In [ ]:
run_agent("Calculate 5000 * 0.15 and tell me the current date and time.")

# Part 4: Creating a ReAct AI Agent

## TASK 7: ReAct Aent Overview ( Conceptual )

1. What is ReAct?
- ReAct stands for Reason + Act.
- A ReAct agent combines the LLM's reasoning with actions performed through tools.

2. Why ReAct agents are powerful?
- ReAct agents are powerful because they can dynamically decide which tools to use and perform multiple actions to solve a problem.

## Task 8: Build a ReAct Agent

In [61]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langchain.agents import create_agent

In [62]:
load_dotenv()

True

In [63]:

llm = ChatOpenAI(model="gpt-4o-mini",temperature=0)

In [64]:
@tool
def calculator(expression: str) -> str:
    """Perform a mathematical calculation."""

    try:
        result = eval(expression)
        return str(result)

    except Exception as e:
        return f"Calculation error: {e}"

In [65]:
@tool
def company_policy_lookup(query: str) -> str:
    """Look up company policy information."""

    policies = {
        "leave": "Employees receive 20 paid leaves per year.",
        "work from home": "Employees can work from home 2 days per week.",
        "working hours": "Working hours are 9 AM to 6 PM.",
        "notice period": "The standard notice period is 30 days."
    }

    query = query.lower()

    for key, value in policies.items():

        if key in query:
            return value

    return "No matching policy found."

In [66]:

wikipedia = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(
        top_k_results=2,
        doc_content_chars_max=1500
    )
)

In [67]:

tools = [
    calculator,
    company_policy_lookup,
    wikipedia
]


In [68]:
system_prompt = """
You are a helpful AI assistant.

You have access to tools for:
1. Mathematical calculations
2. Company policy lookup
3. Wikipedia information

Choose a tool when external information or calculation is required.

Follow this general process:

Think about what needs to be done.
Act by selecting an appropriate tool.
Observe the tool result.
Continue if another tool is required.
Give the final answer when the task is complete.

Do not use a tool when it is unnecessary.
"""

In [88]:

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)


In [89]:
def run_agent(question):
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question
                }
            ]
        }
    )
    for message in result["messages"]:
        if hasattr(message, "tool_calls") and message.tool_calls:

            for call in message.tool_calls:
                print(call["name"])
                print(call["args"])

        # Tool output
        if message.__class__.__name__ == "ToolMessage":
            print(message.content)

    # Final message
    final_message = result["messages"][-1]
    print(final_message.content)

## TASK 9: Testing the ReAct Agent

In [91]:
run_agent(
    "Who is Alan Turing?"
)


wikipedia
{'query': 'Alan Turing'}
Page: Alan Turing
Summary: Alan Mathison Turing (; 23 June 1912 – 7 June 1954) was an English mathematician, computer scientist, logician, cryptanalyst, philosopher and theoretical biologist. He was highly influential in the development of theoretical computer science, providing a formalisation of the concepts of algorithm and computation with the Turing machine, which can be considered a model of a general-purpose computer. Turing is widely considered to be the father of theoretical computer science.
Born in London, Turing was raised in southern England. He graduated from King's College, Cambridge, and in 1938, earned a doctorate degree from Princeton University. During World War II, Turing worked for the Government Code and Cypher School at Bletchley Park, Britain's codebreaking centre that produced Ultra intelligence. He led Hut 8, the section responsible for German naval cryptanalysis. Turing devised techniques for speeding the breaking of German 

In [ ]:
run_agent("Calculate 1250 * 24.")

In [ ]:
run_agent("What is 15% of 2500 and explain what the result means?")

# Part 5: Mini Project AI Agent Assistant

## Task 10: Agent Use Case:


In [94]:
from dotenv import load_dotenv

from langchain.agents import create_agent
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper

In [95]:
@tool
def calculator(expression: str) -> str:
    """
    Perform mathematical calculations.
    """

    try:
        result = eval(expression)
        return str(result)

    except Exception as e:
        return f"Calculation error: {e}"

In [96]:
@tool
def company_policy_lookup(query: str) -> str:
    """
    Retrieve company policy information.
    """

    policies = {
        "leave": "Employees receive 20 paid leaves per year.",

        "work from home":
            "Employees can work from home 2 days per week.",

        "working hours":
            "Standard working hours are 9 AM to 6 PM.",

        "notice period":
            "The standard notice period is 30 days."
    }

    query = query.lower()

    for key, value in policies.items():

        if key in query:
            return value


In [97]:
wikipedia = WikipediaQueryRun(
    api_wrapper=WikipediaAPIWrapper(
        top_k_results=2,
        doc_content_chars_max=1500
    )
)

In [98]:
tools = [
    calculator,
    company_policy_lookup,
    wikipedia
]

In [99]:
system_prompt = """
You are an AI Assistant.

You can help users with:

1. General questions
2. Mathematical calculations
3. Company policy questions
4. Factual information using Wikipedia

Tool selection rules:

- Use calculator for mathematical calculations.
- Use company_policy_lookup for company policy questions.
- Use Wikipedia for factual information that requires external knowledge.
- Do not use a tool when it is unnecessary.

After using a tool, use its result to provide
a clear and concise final answer.
"""

In [100]:
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=system_prompt
)

In [106]:
def ask_assistant(question):
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question
                }
            ]
        }
    )
    for message in result["messages"]:
        if hasattr(message, "tool_calls") and message.tool_calls:

            for call in message.tool_calls:
                print(call["name"])
                print(call["args"])

        # Tool output
        if message.__class__.__name__ == "ToolMessage":
            print(message.content)

    # Final message
    final_message = result["messages"][-1]
    print(final_message.content)

In [107]:
ask_assistant("What is 25 multiplied by 40?")


calculator
{'expression': '25 * 40'}
1000
25 multiplied by 40 is 1000.


In [ ]:
ask_assistant("What is the company work from home policy?")

In [ ]:


ask_assistant("Who is Alan Turing?")

In [ ]:


ask_assistant("Calculate 15% of 2500 and then subtract that amount from 2500.")

## Task 11: Observations and Insights

1. Benefits of tool-augmented agents
- Tool-augmented agents extend the capabilities of an LLM.
- Perform accurate calculations.
- Retrieve current or external information.
- Query databases.
- Automatically choose appropriate tools.

2. Challenges with agents
- Incorrect tool selection.
- Incorrect tool arguments.
- Tool execution failures.
- Higher latency because multiple LLM/tool calls may be required.
- Higher cost from multiple model calls.
- Difficult debugging in complex workflows.
- Agents can take unnecessary actions if the instructions are poorly designed.

3. Difference between chains and agents
- Chain: steps are predefined, predictable exectution, easier to debug
- Agent: steps can be dynamically selected, dynamic exectution, more difficult to debug

4. When to use Agents over RAG
- Use RAG when: "Find relevant information from a known knowledge base."
- Use an Agent when: The problem requires: "Decide what action or tool should be used."